In [10]:
import pandas as pd
import numpy as np
import warnings
import psycopg2
from scipy import stats
from scipy.stats import chi2_contingency
warnings.filterwarnings("ignore")

In [2]:
from config.settings import settings

In [3]:
conn = psycopg2.connect(
    host = settings.POSTGRES_HOST,
    port = settings.POSTGRES_PORT,
    database = settings.POSTGRES_DB_NAME,
    user = settings.POSTGRES_USER,
    password = settings.POSTGRES_PASSWORD
)

# Test 1: Independent Samples Test

## Business Questions

In the banking industry, **account balance** is one of the most critical indicators of customer value. The business wants to understand:

> **Do customers who leave the bank (churn) differ in account balance compared to loyal customers?**

The answer to this question directly influences several strategic decisions, including:

* Retention campaign targeting
* VIP protection programs
* Risk segmentation
* Revenue forecasting

### Business Interpretation

* **If high-balance customers are more likely to churn:** There is a significant risk of losing high-value clients, requiring immediate retention and protection strategies.
* **If low-balance customers churn more frequently:** This may indicate issues with customer engagement, product fit, or the overall value proposition.

---

## Statistical Hypotheses

We aim to compare the **average account balance** between two customer groups:

1. Customers who have churned
2. Customers who have been retained (loyal)

### Null Hypothesis ($H_0$)

The mean account balance is equal between the two groups.

$$
H_0: \mu_{\text{Churned}} = \mu_{\text{Retained}}
$$

### Alternative Hypothesis ($H_1$)

The mean account balance differs between the two groups.

$$
H_1: \mu_{\text{Churned}} \neq \mu_{\text{Retained}}
$$

This is a **two-tailed hypothesis test**, since the objective is to determine whether a difference exists, regardless of its direction (higher or lower).

---

## Column Selection

To perform this hypothesis test, only two columns are required:

| Column    | Type                 | Purpose                                               |
| --------- | -------------------- | ----------------------------------------------------- |
| `Balance` | Continuous numerical | Represents the account balance being compared         |
| `Churned` | Binary categorical   | Defines the two customer groups: churned vs. retained |

---

## Test Assumptions

The **Independent Samples t-test** relies on several important assumptions:

### 1. Independence

Observations must be independent of one another.

In banking datasets, each customer represents a unique record, so this assumption is generally satisfied.

### 2. Normality

The distribution of `Balance` within each group should be approximately normal.

If the sample size is large (**n > 30 per group**), the t-test is considered relatively robust to moderate violations of normality due to the **Central Limit Theorem**.

### 3. Homogeneity of Variance

The variance of `Balance` should be equal across the two groups.

This assumption can be tested using **Levene's Test**.

* If variances are equal → use the standard Independent Samples t-test.
* If variances are unequal → use **Welch's t-test**.

---

## Why This Test?

The choice of statistical test is based on the structure of the variables and the analytical objective:

* **Dependent variable:** Numerical (`Balance`)
* **Independent variable:** Binary categorical (`Churned`)
* **Objective:** Compare the means of two independent groups

Therefore, an **Independent Samples t-test** is appropriate for determining whether the average account balance differs significantly between churned and retained customers.


### Implement Test 
#### Step 1 - Data Preparation

In [4]:
query = """
SELECT
    a.Balance,
    d.Churned
FROM account a
JOIN demographic d
    ON a.CustomerId = d.CustomerId
"""

df_t_test = pd.read_sql(query, conn)

In [5]:
df_t_test.head()

,balance,churned
0,119827.49,True
1,83807.86,False
2,159660.80,True
3,119827.49,False
4,125510.82,False


In [6]:
df_t_test.info

<bound method DataFrame.info of         balance  churned
0     119827.49     True
1      83807.86    False
2     159660.80     True
3     119827.49    False
4     125510.82    False
...         ...      ...
9995  119827.49    False
9996   57369.61    False
9997  119827.49     True
9998   75075.31     True
9999  130142.79    False

[10000 rows x 2 columns]>

#### Step 2 - Split Groups

In [8]:
churned_t_test = df_t_test[df_t_test["churned"] == 1]["balance"]
retained_t_test = df_t_test[df_t_test["churned"] == 0]["balance"]

#### Step 3 - Descriptive Stats

In [9]:
print("Churned Mean : ", churned_t_test.mean())
print("Retained Mean : ", retained_t_test.mean())

Churned Mean :  120521.27620520373
Retained Mean :  119650.01690066556


#### Step 4 - Variance Check

##### If p > 0.05, equal variance assumed

In [11]:
stats.levene(churned_t_test, retained_t_test)

LeveneResult(statistic=np.float64(36.69675448689501), pvalue=np.float64(1.4298234841363054e-09))

##### Since, p-value is less than 0.05, H0 is rejected. So variances are not equal.

#### Step 5 — Run t-test
##### Since, the variances are not equal, we should used Welch’s t-test.

In [13]:
t_stats, p_value = stats.ttest_ind(
    churned_t_test, 
    retained_t_test,
    equal_var = False
)

print("t-stat", t_stats)
print("p_value", p_value)

t-stat 1.3534724523393942
p_value 0.17601045694361078


#### Output Interpretation
#### The p-value is more than 0.05. It means we fail to reject null hypothesis of test. So, final interpretation is that There is no difference between balance amount for churned and retained people. It shows, churining of clients is not caused by amount of balance account.

# Test 2: Chi-Square Test of Independence

## Business Questions

In the banking industry, one of the strongest signals of churn risk is **Account Activity / Customer Engagement Level**. The business wants to understand:

> **Are inactive customers more likely to churn?**

If a dependency exists, the insight can directly support several strategic initiatives:

* Early Warning System development
* Reactivation campaign design
* Customer Health Score modeling

If customer activity is significantly associated with churn:

* Engagement becomes a key driver of churn.
* Behavior-based retention strategies will be required to reduce attrition risk.

---

## Statistical Hypotheses

We are examining the relationship between two categorical variables:

* `Churned` → Yes / No
* `IsActive` → Active / Inactive

### Null Hypothesis ($H_0$)

Churn and customer activity are independent.

$$
H_0: \text{Churned and IsActive are independent}
$$

### Alternative Hypothesis ($H_1$)

Churn and customer activity are associated (dependent).

$$
H_1: \text{Churned and IsActive are associated}
$$

---

## Column Selection

To perform this hypothesis test, only two columns are required:

| Column     | Description                                                     |
| ---------- | --------------------------------------------------------------- |
| `IsActive` | Represents whether the customer's account is active or inactive |
| `Churned`  | Defines the two customer groups: churned vs. retained           |

---

## Contingency Table Concept

The **Chi-Square Test of Independence** operates on a frequency (**contingency**) table.

It evaluates whether the observed frequencies across categorical groups differ significantly from the frequencies we would expect if the variables were independent.

For example:

|          |       Churned = No |      Churned = Yes |
| -------- | -----------------: | -----------------: |
| Active   | Observed frequency | Observed frequency |
| Inactive | Observed frequency | Observed frequency |

The test then compares the **observed frequencies** with the **expected frequencies** under the assumption that `Churned` and `IsActive` are independent.

---

## Test Assumptions

To ensure the validity of the Chi-Square Test, the following assumptions must be satisfied:

### 1. Variables Are Categorical

Both variables must be categorical.

* `IsActive` → Binary categorical
* `Churned` → Binary categorical

### 2. Independence of Observations

Each observation must be independent.

In the banking dataset, each customer represents a unique record, so the independence assumption holds.

### 3. Expected Frequency > 5

Each cell in the contingency table should have a sufficiently large expected frequency, commonly at least **5**.

If the expected frequencies are too small, **Fisher's Exact Test** can be used as an alternative, particularly for a 2 × 2 contingency table.

---

## Why This Test?

The choice of statistical test is based on the structure of the variables and the analytical objective:

* **Dependent variable:** Categorical (`IsActive`)
* **Independent variable:** Binary categorical (`Churned`)
* **Objective:** Determine whether there is an association between customer activity and churn

Therefore, the **Chi-Square Test of Independence** is appropriate for determining whether customer activity is significantly associated with customer churn.


#### Implement Test
##### Step 1 — Data Preparation

In [14]:
query = """
SELECT
    d.Churned,
    a.IsActive
FROM demographic d
JOIN account a
    ON d.CustomerId = a.CustomerId
"""

df_chi = pd.read_sql(query, conn)

In [15]:
df_chi.head()

,churned,isactive
0,True,True
1,False,True
2,True,False
3,False,False
4,False,True


In [16]:
df_chi.shape

(10000, 2)

In [19]:
df_chi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   churned   10000 non-null  bool 
 1   isactive  10000 non-null  bool 
dtypes: bool(2)
memory usage: 19.7 KB


#### Step 2 — Build Contingency Table

In [20]:
ct = pd.crosstab(df_chi["churned"], df_chi["isactive"])
print(ct)

isactive  False  True 
churned               
False      3547   4416
True       1302    735


#### Step 3 — Run Chi-Square Test

In [21]:
chi2, p_value, dof, expected = chi2_contingency(ct)

In [24]:
print("Chi2 : ", chi2)
print("p-value : ", p_value)
print("Degrees of Freedom : ", dof)

Chi2 :  242.98534164287963
p-value :  8.785858269303703e-55
Degrees of Freedom :  1


#### Step 4 — Expected Frequencies for checking assumption

In [25]:
print(expected)

[[3861.2587 4101.7413]
 [ 987.7413 1049.2587]]


### Output Interpretation
### The p-value is less than 0.05. It means we reject null hypothesis of test. So, final interpretation is that Churn and customer activity are associated (dependent)

# Test 3: ANOVA

## Business Questions

The bank operates across multiple countries/regions. The business wants to understand:

> **Does customer tenure differ significantly across countries?**

### Why Is This Question Important?

If tenure is lower in certain countries, it may indicate that:

* Customers in those markets tend to leave earlier or maintain shorter relationships with the bank.
* Service quality, competition, product offerings, or digital experience may be weaker in those regions.
* Customer acquisition and attrition dynamics differ by market.

The results of this analysis can directly inform strategic decisions such as:

* Designing country-specific retention strategies.
* Increasing investment in products or channels in markets with lower tenure.
* Benchmarking regional and branch performance across countries.

---

## Statistical Hypotheses

We are comparing the **average customer tenure across multiple countries**.

### Null Hypothesis ($H_0$)

The average tenure is the same across all countries.

$$
H_0: \mu_1 = \mu_2 = \mu_3 = \cdots = \mu_k
$$

### Alternative Hypothesis ($H_1$)

The average tenure is not the same across all countries.

$$
H_1: \text{At least one country's mean tenure differs}
$$

> **Important:** ANOVA tells us whether at least one group mean is significantly different. It does not tell us which specific countries differ. If the ANOVA is significant, a **post-hoc test such as Tukey's HSD** can be used to identify the specific group differences.

---

## Column Selection

To perform this hypothesis test, only two columns are required:

| Column      | Type              | Description                                                         |
| ----------- | ----------------- | ------------------------------------------------------------------- |
| `Tenure`    | Numerical         | Represents how long the customer has been using the bank's services |
| `Geography` | Multi-categorical | Represents the different countries/locations                        |

---

## Test Assumptions

To ensure the validity of the ANOVA test, the following assumptions should be evaluated:

### 1. Independence

Each customer must represent an independent observation.

In customer datasets, this assumption is typically satisfied because each customer is recorded once.

### 2. Normality Within Each Group

The distribution of `Tenure` within each country should be approximately normal, or the sample sizes should be sufficiently large.

In practice, ANOVA is relatively robust to moderate violations of normality, particularly when each group has a sufficiently large number of observations.

### 3. Homogeneity of Variances

The variance of `Tenure` should be reasonably similar across countries.

This assumption can be assessed using **Levene's Test**.

* If variances are approximately equal → use standard **One-Way ANOVA**.
* If variances are significantly different → use **Welch's ANOVA**.

### 4. Sample Size per Group

Each country should have an adequate number of observations to provide reliable estimates of the group mean and variance.

---

## Why This Test?

The choice of statistical test is based on the structure of the variables and the analytical objective:

* **Dependent variable:** Numerical (`Tenure`)
* **Independent variable:** Multi-categorical (`Geography`)
* **Number of groups:** More than two countries
* **Objective:** Compare the average tenure across multiple countries

Therefore, **One-Way ANOVA** is appropriate for determining whether the mean customer tenure differs significantly across countries.

If the ANOVA result is statistically significant, a **post-hoc test** should be performed to determine which specific countries have significantly different mean tenure values.


### Implement Test
#### Step 1 — Data Preparation

In [26]:
query = """
SELECT
    l.Geography AS Country,
    a.Tenure
FROM account a
JOIN demographic d
    ON a.CustomerId = d.CustomerId
JOIN location l
    ON d.LocationId = l.LocationId
WHERE a.Tenure IS NOT NULL
  AND l.Geography IS NOT NULL;"""

df_anova = pd.read_sql(query, conn)

In [27]:
df_anova.head()

,country,tenure
0,France,2
1,USA,1
2,UK,8
3,Germany,1
4,UK,2


In [28]:
df_anova.shape

(10000, 2)

In [29]:
df_anova.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   country  10000 non-null  object
 1   tenure   10000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 156.4+ KB


#### Step 2 — Descriptive stats

In [30]:
summary = df_anova.groupby("country")["tenure"].agg(["count", "mean", "std"])
print(summary.sort_values("mean", ascending = False))

         count      mean       std
country                           
UK        1703  5.127422  2.939195
France    1714  5.042590  2.904582
USA       1669  5.002397  2.890694
Germany   1626  4.987700  2.870085
Canada    1679  4.982132  2.888003
Spain     1609  4.927906  2.857046


### Step 3 — Check variance equality
#### If p > 0.05, equal variance assumed

In [32]:
groups = [g["tenure"].values for _,g in df_anova.groupby("country")]
lev = stats.levene(*groups)

In [33]:
print(lev)

LeveneResult(statistic=np.float64(0.5338267546333629), pvalue=np.float64(0.7508283480014718))


#### Since, p-value is more than 0.05, H0 is rejected. So variances are not equal.

#### Step 4 — Run test

In [34]:
f_stat, p_value_anova = stats.f_oneway(*groups)
print("F : ", f_stat)
print("p-value : ", p_value_anova)

F :  0.91514100654397
p-value :  0.46986633791017507


### Output Interpretation
#### The p-value is greater than 0.05. It means we fail to reject null hypothesis of test. So, final interpretation is that The average of tenure is all countires are the same

# Test 4: Effect Size

So far, the results of our statistical tests are:

* **Welch's t-test:** No significant difference in mean account balance between churned and retained customers.
* **Chi-Square Test:** There is a significant association between account activity and customer churn.
* **ANOVA:** No significant difference in mean tenure across different countries.

The statistical significance of a test tells us **whether an effect or difference exists**, but it does not necessarily tell us **how large or practically important that effect is**.

Therefore, we use **effect size** to measure the magnitude of the observed difference or association.

---

## Cohen's d: Effect Size for the t-Test

For the comparison of account balance between churned and retained customers, **Cohen's d** can be used to measure the magnitude of the difference between the two group means.

A simplified formula is:

$$
d = \frac{\bar{X}_1 - \bar{X}*2}{S*{\text{pooled}}}
$$

Where:

* $\bar{X}_1$ = Mean balance of the first group
* $\bar{X}_2$ = Mean balance of the second group
* $S_{\text{pooled}}$ = Pooled standard deviation of the two groups

Cohen's d focuses on the **practical magnitude** of the difference rather than simply whether the difference is statistically significant.

### Interpretation of Cohen's d

| **Cohen's d** | **Effect Size** |
| ------------: | --------------- |
|           0.2 | Small           |
|           0.5 | Medium          |
|           0.8 | Large           |
|          1.2+ | Very Large      |

### Interpretation

For example:

* **d ≈ 0.2:** The difference between churned and retained customers is small.
* **d ≈ 0.5:** The difference is moderate.
* **d ≈ 0.8:** The difference is large and potentially practically important.
* **d ≥ 1.2:** The difference is very large.

> **Important:** A statistically non-significant Welch's t-test means there is insufficient evidence that the population means differ. Cohen's d can still be calculated to describe the observed magnitude, but it should be interpreted together with the confidence interval and statistical test result.

---

## Why Effect Size Matters

Statistical significance and practical significance are different concepts.

A very large dataset can produce a statistically significant result for a very small effect, while a meaningful effect may fail to reach statistical significance in a small sample.

Therefore, effect size helps the business answer:

> **"Even if there is a difference, is that difference large enough to matter from a business perspective?"**

For the current analysis, Cohen's d is particularly useful for quantifying the practical difference in **account balance between churned and retained customers**.


In [36]:
# means
churned_t_test_mean = churned_t_test.mean()
retained_t_test_mean = retained_t_test.mean()

# stds
churned_t_test_std = churned_t_test.std()
retained_t_test_std = retained_t_test.std()

# sample size
churned_t_test_n = len(churned_t_test)
retained_t_test_n = len(retained_t_test)

# pooled std
pooled_std = np.sqrt(
    ((churned_t_test_n - 1) * churned_t_test_std ** 2 + (retained_t_test_n - 1) * retained_t_test_std ** 2) / (churned_t_test_n + retained_t_test_n - 2)
)

# cohen d 
cohen_d = (churned_t_test_mean - retained_t_test_mean) / pooled_std

In [37]:
print("Cohen's d : ", cohen_d)

Cohen's d :  0.03623904081079434


The **Independent Samples t-test** indicated no statistically significant difference in average account balance between churned and retained customers, as evidenced by the high p-value.

Additionally, the effect size (**Cohen's d = 0.03**) is negligible, suggesting that account balance has **minimal practical impact** on customer churn behavior.

| **Metric**                 | **Interpretation**                                                                                                                        |
| -------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------- |
| **High p-value in t-test** | There is insufficient statistical evidence to conclude that the average account balance differs between churned and retained customers.   |
| **Small Cohen's d (0.03)** | The observed difference is negligible in practical terms, indicating that account balance has little meaningful impact on churn behavior. |

### Business Conclusion

The analysis suggests that **account balance should not be considered a major standalone factor for churn prediction or retention targeting**. The bank may achieve better results by focusing on variables that show stronger statistical associations and practical effects, such as customer activity and engagement.


#### **Cramér's V: Chi-Square Effect Size**

While the **Chi-Square Test of Independence** tells us whether there is a statistically significant association between two categorical variables, **Cramér's V** measures the **strength of that association**.

In our analysis, it helps determine how strongly **customer activity (`IsActive`) is associated with customer churn (`Churned`)**.

### Interpretation of Cramér's V

| **Cramér's V** | **Effect Size** |
| -------------: | --------------- |
|            0.1 | Weak            |
|            0.3 | Moderate        |
|            0.5 | Strong          |

Generally, Cramér's V ranges from **0 to 1**:

* **0** → No association
* **Around 0.1** → Weak association
* **Around 0.3** → Moderate association
* **Around 0.5 or higher** → Strong association

### Business Interpretation

A statistically significant Chi-Square result indicates that **customer activity and churn are associated**. Cramér's V complements this result by showing **how strong that association is**.

For example, if the analysis produces a Cramér's V close to **0.1**, the relationship is statistically significant but relatively weak from a practical perspective. If the value is closer to **0.5**, customer activity has a much stronger relationship with churn and may be an important factor for retention strategies.

> **Key takeaway:** The Chi-Square test answers **"Is there an association?"**, while Cramér's V answers **"How strong is the association?"**


In [38]:
n_chi = ct.sum().sum()
k_chi = min(ct.shape)

cramers_v = np.sqrt(chi2 / (n_chi * (k_chi - 1)))

In [42]:
print("Cramer's V : ", cramers_v)

Cramer's V :  0.15587987094005423


The **Chi-Square test** shows a statistically significant relationship between customer activity and churn. However, **Cramér's V = 0.15** indicates a **weak association**, suggesting that while customer engagement is related to churn, it is not the sole driving factor.

| **Metric**                           | **Interpretation**                                                                                                                                    |
| ------------------------------------ | ----------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Small p-value in Chi-Square test** | There is a statistically significant relationship between account activity and customer churn.                                                        |
| **Cramér's V = 0.15**                | The strength of the association is weak, indicating that customer activity is related to churn but is not the only factor driving customer attrition. |

### Business Conclusion

Customer activity is a **statistically significant but relatively weak predictor of churn**. Therefore, the bank should consider engagement when developing retention strategies, but should combine it with other customer characteristics and behavioral factors rather than relying on account activity alone.


#### **Eta / Omega: ANOVA Effect Size**

While **ANOVA** determines whether there is a statistically significant difference in mean tenure across countries, an **effect size** helps us understand the **magnitude of that difference**.

For ANOVA, commonly used effect-size measures include **Eta-squared ($\eta^2$)** and **Omega-squared ($\omega^2$)**.

* **Eta-squared ($\eta^2$):** Estimates the proportion of total variance in `Tenure` explained by `Geography`.
* **Omega-squared ($\omega^2$):** Provides a less biased estimate of the population effect size and is often preferred when reporting the practical impact of ANOVA results.

### Interpretation of Eta / Omega

| **Eta / Omega Effect Size** | **Interpretation** |
| --------------------------: | ------------------ |
|                        0.01 | Small              |
|                        0.06 | Medium             |
|                        0.14 | Large              |

### Business Interpretation

A small effect size indicates that **Geography explains only a small proportion of the variation in customer tenure**, even if the ANOVA happens to produce a statistically significant result.

A large effect size would indicate that differences between countries account for a meaningful proportion of the variation in customer tenure and could justify country-specific business strategies.

> **Key takeaway:** ANOVA answers **"Is there a difference in mean tenure?"**, while Eta-squared or Omega-squared answers **"How much of the variation in tenure is explained by country?"**


In [43]:
# calculate SS
all_values = df_anova["tenure"].values
grand_mean = np.mean(all_values)

ss_between = sum(
    len(group) * (np.mean(group) - grand_mean) ** 2
    for group in groups
)

ss_within = sum(
    sum((group - np.mean(group)) ** 2)
    for group in groups
)

ss_total = ss_between + ss_within

In [44]:
print("SS_total : ", ss_total)

SS_total :  83638.36159999993


In [45]:
# Calculate ETA Score
k = len(groups)
n = len(all_values)

df_between = k - 1
df_within = n - k

ms_within = ss_within / df_within

eta_squared = ss_between / ss_total

In [46]:
print("Eta Squared(η²) :", eta_squared)

Eta Squared(η²) : 0.00045763568409215805


In [47]:
# Calculate Omega Score
omega_squared = (
    ss_between - (df_between * ms_within)
) / (ss_total + ms_within)

In [48]:
print("Omega Squared (ω²):", omega_squared)

Omega Squared (ω²): -4.243129706659199e-05


The **One-Way ANOVA test** indicates no statistically significant difference in average customer tenure across countries (**p-value > 0.05**).

The effect size measures further support this finding, with **Eta Squared ($\eta^2$) = 0.00046** and **Omega Squared ($\omega^2$) = 0**, indicating a **negligible practical impact**.

This suggests that **geographic region does not meaningfully influence customer tenure** in this dataset.

| **Metric**                      | **Interpretation**                                                                                        |
| ------------------------------- | --------------------------------------------------------------------------------------------------------- |
| **High p-value in ANOVA test**  | There is no statistically significant difference in average customer tenure across countries.             |
| **Small Eta/Omega effect size** | Any observed differences in tenure across countries are negligible from a practical business perspective. |

### Business Conclusion

The analysis suggests that **geographic region should not be considered a major factor when developing customer tenure or retention strategies**. Instead, the bank should focus on other customer characteristics and behavioral factors that may have a stronger relationship with customer retention.
